定义模型

In [11]:

from pydoc_data.topics import topics
from statistics import multimode

from anyio.lowlevel import checkpoint
from langchain.chat_models import init_chat_model
import os

from openai import api_key
from openai.types.admin.organization.projects.service_account_create_response import APIKey

model = init_chat_model(
    model="deepseek-v4.1-flash",
    base_url="https://cn-beijing.maas.qianwenaiapi.com/compatible-mode/v1",
    api_key=os.getenv('DASHSCOPE_API_KEY'))


定义工具

In [12]:
from langchain_tavily import TavilySearch
from dotenv import load_dotenv

load_dotenv()
web_search = TavilySearch(
    max_results=5,
    topic = "general"
)


添加记忆管理

In [13]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

#链接sqlite
connection = sqlite3.connect("AI_private_chief.db",check_same_thread=False)
#初始化checkpointer
checkpointer = SqliteSaver(connection)
#自动建表
checkpointer.setup()

定义只能体

In [14]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer,
)

测试

In [15]:
from langchain.messages import  HumanMessage
multimodel_messages = HumanMessage([
    {"type":"text","text":"帮我看看能做什么"},
    {"type":"image","url":'https://pic.nximg.cn/file/20230530/9629334_132255638101_2.jpg'}
])

config = {'configurable':{"thread_id":"1"}}

response = agent.invoke({'messages':[multimodel_messages]},config)

In [16]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://pic.nximg.cn/file/20230530/9629334_132255638101_2.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://pic.nximg.cn/file/20230530/9629334_132255638101_2.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://pic.nximg.cn/file/20230530/9629334_132255638101_2.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://pic.nximg.cn/file/20230530/9629334_132255638101_2.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https

In [17]:
response = agent.invoke({'messages':[HumanMessage(content="我喜欢第七道菜，详细说说")]},config)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://pic.nximg.cn/file/20230530/9629334_132255638101_2.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://pic.nximg.cn/file/20230530/9629334_132255638101_2.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://pic.nximg.cn/file/20230530/9629334_132255638101_2.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://pic.nximg.cn/file/20230530/9629334_132255638101_2.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https